### Data Cleaning

Load the datasets using pandas

In [ ]:
import pandas as pd

weather = pd.read_csv('../data/raw/weather_daily.csv', 
                            na_values=['NA', ''])
soil = pd.read_csv('../data/raw/soil_sensor_data.csv', 
                              na_values=['NA', ''])
params = pd.read_csv('../data/raw/crop_zone_parameters.csv', 
                                  na_values=['NA', ''])


print("Crop Zone Parameters:")
print(params.head())
print(params.info())

print("\nSoil Sensor Data:")
print(soil.head())
print(soil.info())

print("\nWeather Daily:")
print(weather.head())
print(weather.info())

# Identify specific missing values
print("Missing values in soil sensor:")
print(soil[soil.isna().any(axis=1)])

print("\nMissing values in weather:")
print(weather[weather.isna().any(axis=1)])

# Describe statistics to look for outliers
print("\nSoil sensor summary statistics:")
print(soil.describe())

print("\nWeather daily summary statistics:")
print(weather.describe())

# Check for sensor status issues
print("\nSensor status unique values and counts:")
print(soil['sensor_status'].value_counts())
print("\nRows with sensor status not OK:")
print(soil[soil['sensor_status'] != 'OK'])

# Let's inspect the temperature outlier
print("Temperature outlier row:")
print(weather[weather['temperature_c'] > 35])

# Let's inspect the tank level outlier
print("\nTank level outlier row:")
print(soil[soil['tank_level_liters'] > 5000])

# Let's check for any other zero-flow or weird combinations
print("\nPump drawing power but zero flow:")
print(soil[(soil['pump_power_watts'] > 100) & (soil['pump_flow_lpm'] == 0)])

--- Crop Zone Parameters ---
  zone_id crop_type  area_m2  min_moisture_pct  target_moisture_pct  \
0  Zone_A    tomato      120                22                   33   
1  Zone_B      kale       90                24                   35   
2  Zone_C     maize      180                20                   31   

   field_capacity_pct  drainage_coefficient  
0                  41                  0.18  
1                  43                  0.15  
2                  40                  0.22  
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   zone_id               3 non-null      str    
 1   crop_type             3 non-null      str    
 2   area_m2               3 non-null      int64  
 3   min_moisture_pct      3 non-null      int64  
 4   target_moisture_pct   3 non-null      int64  
 5   field_capacity_pct    3 non-null      int64  
 6   

Data cleaning and anomaly detection are critical initial steps in data analysis. 
Using **pandas**, we can easily profile our data to find missing values, 
outliers, unit inconsistencies, and sensor anomalies.

### 1. Identifying Missing Values

Missing values occur when no data is recorded for a specific observation. Pandas 
represents missing numerical or categorical data as `NaN` (Not a Number) or 
`None`.

* **`df.isna()` or `df.isnull()**`: Returns a boolean DataFrame where `True` 
indicates a missing value.
* **`df.isna().sum()`**: Counts the number of missing values in each column.
* **Boolean Indexing**: Used to filter and display the exact rows containing 
missing values.

**Examples in the Data**

* In `soil_sensor_data.csv`, row 16 contains a missing value (`NaN`) for 
`soil_moisture_pct` on `2026-03-06`.
* In `weather_daily.csv`, `rainfall_mm` is missing on `2026-03-08`, and 
`humidity_pct` is missing on `2026-03-21`.


### 2. Identifying Outliers

Outliers are data points that deviate significantly from the rest of the 
dataset. They can represent true extreme events or data entry errors.

* **`df.describe()`**: Generates summary statistics (mean, standard deviation, 
min, max, and quartiles). If the `max` or `min` value is vastly different from 
the $75\text{th}$ or $25\text{th}$ percentiles, an outlier is likely present.
* **Interquartile Range (IQR) Method**: Outliers are identified as values 
falling outside the range:

$$[Q_1 - 1.5 \times IQR, \quad Q_3 + 1.5 \times IQR]$$

where $Q_1$ is the $25\text{th}$ percentile, $Q_3$ is the $75\text{th}$ 
percentile, and $IQR = Q_3 - Q_1$. You can compute these using 
`df['column'].quantile(0.25)` and `df['column'].quantile(0.75)`.
* **Z-Score Method**: Measures how many standard deviations ($\sigma$) a data 
point is from the mean ($\mu$):

$$Z = \frac{x - \mu}{\sigma}$$

Typically, any point with $|Z| > 3$ is flagged as an outlier.



**Examples in the Data**

* In `weather_daily.csv`, running `.describe()` shows a maximum `temperature_c` 
of $45.8^\circ\text{C}$ on `2026-03-14`, whereas the $75\text{th}$ percentile 
is only $26.5^\circ\text{C}$. This indicates an extreme outlier.
* In `soil_sensor_data.csv`, the `tank_level_liters` column hits a maximum value 
of $9900\text{ L}$ on the same day (`2026-03-14`), while the normal operating 
capacity stays between $3300\text{ L}$ and $4800\text{ L}$.


### 3. Identifying Inconsistent Units

Inconsistent units occur when measurements are recorded using different systems 
(e.g., mixing Liters and Gallons, or Celsius and Fahrenheit) or when raw data 
scales change unexpectedly.

* **`df.dtypes`**: Ensures all columns possess the expected numerical or 
categorical data type. Text mixed into a numerical column will force the column 
dtype to be `object`.
* **Domain Validation/Value Ranges**: Checking if values fall outside of 
physically or logically possible ranges for the given unit.
* **Cross-Dataset Ratios**: Analyzing whether metrics scale correctly across 
zones or boundaries.


**Examples in the Data**

* The $45.8^\circ\text{C}$ temperature observation could be a unit mix-up. If 
it were accidentally recorded in Fahrenheit, 
$45.8^\circ\text{F} \approx 7.7^\circ\text{C}$ (which would represent a sudden 
cold snap). If it was meant to be $25.8^\circ\text{C}$, it represents a typo.
* The $9900\text{ L}$ tank level reading might also be an uncalibrated raw 
value or an unscaled metric that missed a decimal point (e.g., 
$4900\text{ L}$ or $3900\text{ L}$).

### 4. Identifying Sensor Anomalies

Sensor anomalies are caused by hardware malfunctions, transmission errors, or 
conflicting operational telemetry (e.g., a device reporting that it is drawing 
power but producing zero output).

* **Categorical Filtering**: Filtering explicit error flags or maintenance 
statuses built into the data schema.
* **Logical Contradiction Rules**: Using conditional boolean indexing (with 
logical operators `&` and `|`) to catch physically impossible combinations 
between multiple sensors.

**Examples in the Data**
* In `soil_sensor_data.csv` at row 61 (`2026-03-21`), the `sensor_status` is 
explicitly flagged as `'CHECK'`.
* By applying cross-sensor logic, pandas reveals that on this row, 
`pump_power_watts` is $468\text{ W}$ (the pump is fully active) but 
`pump_flow_lpm` is $0.0\text{ lpm}$ (no water is moving). This contradiction 
efficiently isolates a hardware anomaly (e.g., a blocked pipe, dry run, or 
broken flow meter).

Clean the dataset

In [ ]:
import pandas as pd
import numpy as np

weather_clean = weather.copy()

# Fix missing continuous values (Rainfall and Humidity) via linear interpolation
weather_clean['rainfall_mm'] = weather_clean['rainfall_mm'].interpolate(method='linear')
weather_clean['humidity_pct'] = weather_clean['humidity_pct'].interpolate(method='linear')

# Isolate, mask, and interpolate the temperature sensor spike (45.8°C)
weather_clean.loc[weather_clean['temperature_c'] > 35, 'temperature_c'] = np.nan
weather_clean['temperature_c'] = weather_clean['temperature_c'].interpolate(method='linear')


soil_clean = soil.copy()

# Impute missing soil moisture within its respective crop zone sequence
soil_clean['soil_moisture_pct'] = soil_clean.groupby('zone_id')['soil_moisture_pct'].transform(
    lambda x: x.interpolate(method='linear')
)

# Identify and mask the arbitrary tank telemetry surge (9900 Liters)
soil_clean.loc[soil_clean['tank_level_liters'] > 5500, 'tank_level_liters'] = np.nan
soil_clean['tank_level_liters'] = soil_clean.groupby('zone_id')['tank_level_liters'].transform(
    lambda x: x.interpolate(method='linear')
)

# Address the logical telemetry contradiction (Pump running at 468W but flow reads 0.0 lpm)
# Calculate the standard operational flow rate for Zone B pumps
zone_b_active_flow = soil_clean[(soil_clean['zone_id'] == 'Zone_B') & (soil_clean['pump_flow_lpm'] > 0)]['pump_flow_lpm'].mean()

# Apply standard flow and adjust flag status on the faulty entry
contradiction_mask = (soil_clean['sensor_status'] == 'CHECK') & (soil_clean['pump_flow_lpm'] == 0)
soil_clean.loc[contradiction_mask, 'pump_flow_lpm'] = round(zone_b_active_flow, 1)
soil_clean.loc[contradiction_mask, 'sensor_status'] = 'CLEANED'


weather_clean.to_csv('../data/processed/weather_daily_cleaned.csv', index=False)
soil_clean.to_csv('../data/processed/soil_sensor_data_cleaned.csv', index=False)

Data processing complete. Outputs generated successfully.
